In [1]:
%load_ext autoreload
%autoreload 2

In [29]:
import warnings

from embedding.hybrid_search import HybridSearch

warnings.filterwarnings("ignore")

from collections import defaultdict
from utils.constants import BATCH_SIZE, NEAREST_NEIGHBOUR_COUNT
from tqdm import tqdm
from pathlib import Path
from chunking import chunk_files
from embedding import FAISS, HybridSearch, BM25_Plus, ReciprocalRankFusion, SearchResult, DocumentStore
from evaluator import Evaluator, SearchQueryData
from utils.embed_input import datadict2embed_input, jsonl2datadict
from pprint import pprint
import json

In [18]:
curr_dir = Path.cwd().parent / "data" / "test-repo"
print(curr_dir)
with open("output.jsonl", "w") as f:
    chunk_files(curr_dir, f)

with open("output.jsonl", "r") as f:
    lines = f.readlines()
    n_chunks = len(lines)
    metadatas = list(map(jsonl2datadict, lines))

target: list[SearchQueryData] = []
with open("../rag_eval_dataset_symbolic.jsonl", "r") as f:
    for line in f.readlines():
        target.append(json.loads(line))


texts: list[str] = []
chunk_ids: list[str] = []
doc_store = DocumentStore()
for i in range(n_chunks):
    embed_input = datadict2embed_input(metadatas[i])
    chunk_ids.append(metadatas[i].chunk_id)
    doc_store.insert(metadatas[i])
    texts.append(embed_input)

dense = FAISS(batch_size=BATCH_SIZE)
bm25 = BM25_Plus()


hybrid_search = HybridSearch([dense, bm25], ReciprocalRankFusion())
hybrid_search.fit(texts, chunk_ids)

/home/gaurav/coding/SoftwareEngineeringAgent/data/test-repo
Embedding and saving batches: 


100%|██████████| 95/95 [00:04<00:00, 21.68it/s]


In [44]:
# q = input("Enter your query: ")
q = "user controller"

hybrid_result = hybrid_search.search(q, 5)
print(hybrid_result)

ids = [res.chunk_id for res in hybrid_result]

res [[SearchResult(chunk_id='fcddce00-5fee-43ac-98b2-3b5ed0c53e0e', score=151.4345703125), SearchResult(chunk_id='000dc661-ea28-43dc-acbb-43d15ab3140a', score=156.7227783203125), SearchResult(chunk_id='eeab3421-4a7c-4004-90c1-0ff0e689b6dc', score=171.11639404296875), SearchResult(chunk_id='6b2b4648-0e04-469c-a189-53bf7f8295e6', score=171.4026336669922), SearchResult(chunk_id='ecbe4e43-0be5-40d0-8f01-c966472e2ec6', score=177.65185546875)]]
res [[SearchResult(chunk_id='fcddce00-5fee-43ac-98b2-3b5ed0c53e0e', score=8.187775300032872), SearchResult(chunk_id='5607e8de-a177-43c7-9eab-666b97356ed2', score=7.2228185531584534), SearchResult(chunk_id='000dc661-ea28-43dc-acbb-43d15ab3140a', score=7.070378006352232), SearchResult(chunk_id='ac9f5b69-43ab-4dd4-b4df-461cdcb4a524', score=6.897009020748821), SearchResult(chunk_id='0dfa5c76-5da9-469f-a7dd-05d23c182485', score=6.882396882456728)]]
Merge outputs [[SearchResult(chunk_id='fcddce00-5fee-43ac-98b2-3b5ed0c53e0e', score=151.4345703125), SearchRe

In [42]:
len(ids)

5

In [33]:
doc_store.get(ids)

[ChunkMetaData(chunk_id='fcddce00-5fee-43ac-98b2-3b5ed0c53e0e', chunk_type='class', name='UserControllersClass', qualified_name='module.UserControllersClass', parent_name='module', parent_type='module', file_path='backend/src/controllers/user_controller.py', file_name='user_controller.py', module_path='backend.src.controllers.user_controller', language='Python', start_line=15, end_line=421, source_code='class UserControllersClass:\n    def __init__(self):\n        self.Config = Settings()\n        self.mongo_db_connection = MongoDBConnection(self.Config.MONGO_URI, self.Config.DB_NAME)\n        self.email_services = EmailServices()\n        self.password_manager = PasswordManager()\n        self.jwt_manager = JWTManager()\n        self.email_regex_parrern = r"^(?!.*\\.\\.)[a-zA-Z0-9!#$%&\'*+/=?^_`{|}~-]+(?:\\.[a-zA-Z0-9!#$%&\'*+/=?^_`{|}~-]+)*@(?:[a-zA-Z0-9-]+\\.)+[a-zA-Z]{2,}$"\n\n    def _start_connection(self):\n        """\n        Start the MongoDB connection.\n        """\n       